# Asistente de Integración Pedidos360 — recorrido

**EP1 · ISY0101 Ingeniería de Soluciones con IA**

Este notebook **no contiene la lógica**: la importa desde `asistente/`, donde vive el
código de verdad. Sirve para recorrer el sistema pieza por pieza y ver qué hace cada una.

El recorrido sigue el camino de una pregunta:

    corpus → ingesta → índices → recuperación → agente → herramientas → guardrails → respuesta

Cada sección dice qué módulo la implementa, por si quieres abrir el archivo.

## 0. Preparación

En Colab clona el repositorio e instala dependencias (tarda unos minutos la primera vez).
En local no hace nada: usa el entorno del curso.

In [1]:
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not Path("Ingenier-a-de-Soluciones-con-Inteligencia-Artificial").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/1samadhi/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial.git"],
                       check=True)
    %pip install -q langchain langchain-classic langchain-groq langchain-community \
                    langchain-huggingface sentence-transformers faiss-cpu scikit-learn python-dotenv
    RAIZ = Path("Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/EP1-asistente-pedidos360")
else:
    RAIZ = Path.cwd()

sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
print(f"Trabajando en: {RAIZ.resolve()}")

Trabajando en: /mnt/D/duoc/2026/2do semestre/Ing. Soluciones con IA/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/EP1-asistente-pedidos360


### Credenciales

`GROQ_API_KEY` es lo único imprescindible. Las de Entra ID solo hacen falta para las
consultas en vivo a la API: **sin ellas el asistente igual funciona** y responde con la
documentación, avisando de que no puede consultar datos actuales.

In [2]:
# En Colab, desde los Secrets (icono de la llave). En local, desde el .env.
if EN_COLAB:
    from google.colab import userdata
    for clave in ("GROQ_API_KEY", "PEDIDOS360_URL", "ENTRA_TENANT_ID", "ENTRA_CLIENT_ID",
                  "ENTRA_APP_ID_URI", "ENTRA_USUARIO_CLIENTE", "ENTRA_PASSWORD_CLIENTE"):
        try:
            os.environ[clave] = userdata.get(clave)
        except Exception:
            pass   # las de Entra son opcionales

from asistente import config

print(f"Modelo:      {config.MODELO}")
print(f"Embeddings:  {config.MODELO_EMBEDDINGS.split('/')[-1]}  (local, sin API)")
print(f"API:         {config.PEDIDOS360_URL}")
print(f"Groq:        {'configurada' if os.getenv('GROQ_API_KEY') else 'FALTA — sin ella no responde'}")
print(f"Entra ID:    {'configurada' if config.hay_credenciales_api() else 'ausente — solo documentación'}")

Modelo:      openai/gpt-oss-120b
Embeddings:  paraphrase-multilingual-MiniLM-L12-v2  (local, sin API)
API:         https://j37oj1wn16.execute-api.us-east-1.amazonaws.com/desarrollo
Groq:        configurada
Entra ID:    configurada


---
## 1. El corpus · `corpus/`

Dos orígenes, etiquetados por separado porque la confianza que merecen es distinta. El
indicador IE3 exige justamente eso: fuentes internas **y** externas.

In [3]:
for origen in ("interno", "externo"):
    archivos = sorted((config.CORPUS / origen).glob("*.md"))
    total = sum(len(a.read_text(encoding="utf-8").split()) for a in archivos)
    print(f"\n{origen.upper()}  ·  {len(archivos)} archivos, {total:,} palabras".replace(",", "."))
    for a in archivos:
        print(f"   {len(a.read_text(encoding='utf-8').split()):>5}  {a.name}")


INTERNO  ·  10 archivos. 7.210 palabras
    1236  00-readme-pedidos360.md
     337  01-cognito.md
     945  02-entra-id.md
     758  03-api-gateway.md
     580  04-despliegue-ec2.md
     637  05-base-de-datos.md
     552  06-frontend-angular.md
     810  07-changelog.md
     479  08-readme-frontend.md
     876  09-errores-frecuentes.md

EXTERNO  ·  3 archivos. 4.114 palabras
     495  owasp-api-top10.md
    2533  rfc6749-oauth2.md
    1086  rfc7519-jwt.md


---
## 2. Ingesta y troceo · `asistente/ingesta.py`

Se divide primero por encabezado de Markdown y luego por tamaño, **anteponiendo a cada
fragmento su documento y su sección**.

Ese detalle no es cosmético. Un fragmento sacado de la mitad de `03-api-gateway.md` no
menciona en su texto ni «API Gateway» ni la sección a la que pertenece, así que su vector
no se parece a una pregunta que use esas palabras. Con el encabezado dentro, sí.

In [4]:
from asistente.ingesta import cargar_documentos, trocear

docs = cargar_documentos()
trozos = trocear(docs)
print(f"{len(docs)} documentos -> {len(trozos)} fragmentos\n")

# Un fragmento cualquiera, para ver el encabezado que se le antepone
ej = next(t for t in trozos if t.metadata["archivo"] == "03-api-gateway.md")
print("EJEMPLO DE FRAGMENTO")
print("-" * 70)
print(ej.page_content[:420])

13 documentos -> 145 fragmentos

EJEMPLO DE FRAGMENTO
----------------------------------------------------------------------
AWS API Gateway como API Manager — AWS API Gateway como API Manager

# AWS API Gateway como API Manager  
El API Gateway es el unico punto de entrada publico. Ofrece HTTPS, enruta hacia
cada microservicio y aplica el autorizador JWT antes de que la peticion llegue a
la EC2.


### Construir los índices

Dos índices sobre los mismos fragmentos: uno **vectorial** (FAISS, con embeddings locales)
y uno **léxico** (TF-IDF). Si ya existen, esta celda los carga en vez de recalcularlos.

In [5]:
from asistente.ingesta import construir
from asistente.recuperador import cargar_indice

if not (config.INDICE / "index.faiss").exists():
    almacen = construir()          # ~30 s la primera vez
else:
    almacen = cargar_indice()
    print(f"Índice ya construido en {config.INDICE.name}/")

/mnt/D/duoc/2026/2do semestre/Ing. Soluciones con IA/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice ya construido en indice/


---
## 3. Recuperación · `asistente/recuperador.py`

Aquí está la decisión de diseño más interesante del proyecto: **no se busca de una sola
manera**. Hay dos índices sobre los mismos fragmentos y sus resultados se fusionan.

La búsqueda densa entiende sinónimos, pero se pierde con literales que no los tienen:
`401`, `/v1/pedidos`, `ms-auth`. La léxica hace exactamente lo contrario.

Veámoslo con una pregunta donde la diferencia todavía se nota.

In [6]:
from asistente.recuperador import recuperar, recuperar_hibrido

P = "Que emisores de identidad acepta la plataforma?"
# La respuesta está en el README, que los enumera los tres juntos.

print("SOLO SEMÁNTICA")
print("-" * 70)
for doc, _s in recuperar(P, k=5, almacen=almacen):
    print(f"  [{doc.metadata['origen']:7}] {doc.metadata['archivo']}")

print("\nHÍBRIDA  (0,8 léxico / 0,2 denso, tope 2 por archivo)")
print("-" * 70)
for doc in recuperar_hibrido(P, almacen=almacen):
    print(f"  [{doc.metadata['origen']:7}] {doc.metadata['archivo']}")

SOLO SEMÁNTICA
----------------------------------------------------------------------


  [interno] 01-cognito.md
  [interno] 01-cognito.md
  [interno] 02-entra-id.md
  [interno] 02-entra-id.md
  [interno] 01-cognito.md

HÍBRIDA  (0,8 léxico / 0,2 denso, tope 2 por archivo)
----------------------------------------------------------------------
  [interno] 09-errores-frecuentes.md
  [interno] 00-readme-pedidos360.md
  [interno] 01-cognito.md
  [interno] 02-entra-id.md
  [interno] 02-entra-id.md


La densa se va a los documentos de **cada** emisor por separado —Cognito, Entra— porque
semánticamente son lo más parecido a la pregunta. La híbrida encuentra el README, que es
donde están **enumerados los tres**.

Los rankings se fusionan con **Reciprocal Rank Fusion**, y un tope por archivo evita que
un solo documento copie todos los puestos. Los pesos 0,8 / 0,2 no son intuición: salen de
medir sobre las 30 preguntas, y la primera versión con pesos iguales resultaba **peor** que
la densa sola.

> **Una advertencia honesta.** La pregunta que originó todo esto —el token de `ms-auth` que
> recibe 401— hoy la aciertan las dos estrategias. No porque la recuperación mejorara, sino
> porque **se añadió al corpus la guía de errores frecuentes que faltaba**. Esa es la
> lección más útil del proyecto: la recuperación no puede encontrar lo que no está escrito.

---
## 4. Herramientas · `asistente/herramientas.py`

Responden lo que **ningún documento puede responder**, porque depende del estado actual
del sistema. Si la API no contesta, lo dicen: nunca devuelven una cifra inventada.

In [7]:
from asistente import herramientas

print(herramientas.consultar_catalogo())
print()
print(herramientas.consultar_pedidos())

Catalogo actual (4 productos):
- id 1: Teclado mecanico — $45.990
- id 2: Mouse inalambrico — $19.990
- id 3: Monitor 27 pulgadas — $189.990
- id 4: Audifonos con cancelacion de ruido — $89.990



Pedidos del comercio: 56 en total, $6.156.710 facturados.

Ranking por facturacion (de mayor a menor):
  1. Monitor 27 pulgadas: $1.709.910 (9 unidades en 6 pedidos)
  2. Audifonos con cancelacion de ruido: $1.529.830 (17 unidades en 9 pedidos)
  3. Teclado mecanico: $1.517.670 (33 unidades en 16 pedidos)
  4. Mouse inalambrico: $1.399.300 (70 unidades en 25 pedidos)

Ranking por cantidad de pedidos (de mayor a menor):
  1. Mouse inalambrico: 25 pedidos
  2. Teclado mecanico: 16 pedidos
  3. Audifonos con cancelacion de ruido: 9 pedidos
  4. Monitor 27 pulgadas: 6 pedidos

El que MAS FACTURA es Monitor 27 pulgadas con $1.709.910.
El MAS PEDIDO es Mouse inalambrico con 25 pedidos.
Usa estos rankings tal como estan; no los recalcules.


---
## 5. El agente decide · `asistente/agente.py`

El enrutamiento **no está cableado con reglas**. Se declaran tres herramientas
—`buscar_documentacion`, `consultar_pedidos`, `consultar_catalogo`— y el modelo elige.

Tres preguntas que ejercitan los tres caminos. La tercera es la que justifica usar un
agente y no un RAG plano: necesita documentación **y** datos actuales a la vez.

In [8]:
from asistente.agente import crear_agente, preguntar

ejecutor = crear_agente()

PREGUNTAS = [
    ("DOCUMENTAL",  "Por que mi token de ms-auth da 401 en /v1/pedidos?"),
    ("DATOS VIVOS", "Cuantos pedidos lleva mi comercio y cual factura mas?"),
    ("MIXTA",       "Como obtengo el listado de pedidos por API y cuantos llevo ahora?"),
]

for etiqueta, p in PREGUNTAS:
    print("=" * 74)
    print(f"[{etiqueta}]  {p}")
    print("-" * 74)
    print(preguntar(p, ejecutor))
    print()

[DOCUMENTAL]  Por que mi token de ms-auth da 401 en /v1/pedidos?
--------------------------------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

El 401 ocurre porque las rutas de negocio (`/v1/pedidos`, `/v1/productos`) están protegidas solo por el autorizador de **Microsoft Entra ID**. Un token emitido por `ms‑auth` es rechazado en el borde del API Gateway, aun cuando sea válido y tenga los claims correctos【1】. Para usar esas rutas debes obtener el token de Entra ID (o configurar un autorizador adicional para `ms‑auth`).

[DATOS VIVOS]  Cuantos pedidos lleva mi comercio y cual factura mas?
--------------------------------------------------------------------------


Tu comercio tiene **56 pedidos** en total, con un monto facturado de **$6.156.710**.  

El producto que **más factura** es **Monitor 27 pulgadas** con **$1.709.910** facturados.

[MIXTA]  Como obtengo el listado de pedidos por API y cuantos llevo ahora?
--------------------------------------------------------------------------


Para obtener el listado de pedidos por API debes llamar al endpoint **GET /v1/pedidos** del microservicio `ms-pedidos` [1]​.

En este momento el comercio tiene **56 pedidos** en total, con un monto facturado de **$6.156.710** [consultar_pedidos].



Con `crear_agente(verboso=True)` se ve el razonamiento: qué herramienta eligió en cada
paso y con qué argumentos la llamó.

---
## 6. Guardrails · `asistente/guardrails.py`

El corpus contiene, legítimamente, datos que un comercio integrado no debe ver:
identificadores de instancia, el tenant de Entra, IP públicas y las credenciales de los
usuarios de prueba. El recuperador puede traer un fragmento que los incluya; lo que no
puede pasar es que salgan en la respuesta.

In [9]:
from asistente.guardrails import sanear

SUCIO = ("Conéctate a la instancia i-0314ddd12fadeb125 en 54.173.206.221, "
         "tenant 5cb85dc6-a73b-41fc-b2b9-5b2a9b3f531b, con admin123.")

limpio, aplicados = sanear(SUCIO)
print("ANTES:  ", SUCIO)
print("\nDESPUÉS:", limpio)
print(f"\n{len(aplicados)} dato(s) filtrado(s)")

# Y no toca el texto legítimo:
inocuo = "En local la API escucha en http://localhost:8082 y el pedido cuesta $45.990."
print("\nTexto inocuo intacto:", sanear(inocuo)[0] == inocuo)

ANTES:   Conéctate a la instancia i-0314ddd12fadeb125 en 54.173.206.221, tenant 5cb85dc6-a73b-41fc-b2b9-5b2a9b3f531b, con admin123.

DESPUÉS: Conéctate a la instancia [id-de-instancia] en [ip-del-servidor], tenant [identificador-interno], con [credencial-omitida].

4 dato(s) filtrado(s)

Texto inocuo intacto: True


---
## 7. Métricas · `evaluacion/`

La recuperación se mide **sin modelo**: se compara qué archivos se recuperan contra los
que el set declara correctos. Es determinista y gratis, así que puede correr las veces
que haga falta.

In [10]:
from evaluacion.evaluar import cargar_set, evaluar_recuperacion

casos = cargar_set()
res = evaluar_recuperacion(casos)

print(f"{len(casos)} preguntas\n")
print(f"{'estrategia':22} {'recall':>8} {'precision':>10} {'sin fuente':>11}")
for nombre, m in res.items():
    print(f"{nombre:22} {m['recall']:>8.2f} {m['precision']:>10.2f} "
          f"{len(m['sin_ninguna_fuente']):>11}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

30 preguntas

estrategia               recall  precision  sin fuente
solo semantica             0.80       0.58           2
hibrida ponderada          0.92       0.45           0


La precisión hay que leerla **contra su techo**, no contra 1,0: con cinco fragmentos y un
tope de dos por archivo, una pregunta con una sola fuente esperada no puede pasar de 2/5.
El máximo alcanzable sobre este set es 0,55.

Fidelidad y relevancia usan un modelo como juez y consumen cuota, así que van aparte:

```bash
python -m evaluacion.evaluar --generacion --n 30
```

---
## Dónde está cada cosa

| Módulo | Líneas | Qué hace | De quién depende |
|---|---:|---|---|
| `config.py` | 54 | Toda la configuración, por variables de entorno | nadie |
| `prompts.py` | 83 | Las tres variantes del prompt de sistema | nadie |
| `guardrails.py` | 36 | Filtra datos sensibles de la salida | nadie |
| `ingesta.py` | 116 | Trocea el corpus y construye los índices | `config` |
| `recuperador.py` | 203 | Búsqueda híbrida y fusión RRF | `config`, `ingesta` |
| `herramientas.py` | 125 | Consultas en vivo a la API con token de Entra | `config` |
| `agente.py` | 77 | Orquesta: declara las herramientas y el modelo elige | todos |

`agente.py` es el único que conoce a los demás; el resto no sabe que el agente existe. Por
eso se pueden probar por separado, y por eso el notebook puede recorrerlos uno a uno.

**Diagramas:** `docs/arquitectura.svg` y `docs/boceto-secuencia-consulta-mixta.svg`.